In [1]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Cole a chave da OpenAI: ").strip()

In [2]:
from typing import List
from dataclasses import dataclass
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv

# Carregar variáveis de ambiente
load_dotenv()

# Definir a estrutura de dependências para recuperação
@dataclass
class RetrievalDependencies:
    documents: List[str]

# Definir o modelo de resultado esperado
class RAGResult(BaseModel):
    answer: str = Field(..., description="Resposta final gerada com base no contexto recuperado")
    retrieved_docs: List[str] = Field(..., description="Lista de documentos relevantes utilizados na resposta")

# Criar o agente RAG
rag_agent = Agent(
    'openai:gpt-6.1-sol',  # Substituído por modelo Groq que você já usa
    deps_type=RetrievalDependencies,
    output_type=RAGResult,
    system_prompt="Você é um assistente que utiliza informações recuperadas para responder com precisão. Use os documentos relevantes para compor sua resposta."
)

# Definir a ferramenta de recuperação de contexto
@rag_agent.tool()  # Adicionei parênteses aqui
async def retrieve_context(ctx: RunContext[RetrievalDependencies], query: str) -> List[str]:
    """
    Recupera documentos relevantes com base na consulta.
    
    Args:
        ctx: Contexto contendo os documentos disponíveis
        query: Consulta para filtrar documentos relevantes
        
    Returns:
        Lista de documentos que correspondem à consulta
    """
    # Busca simplificada: filtra documentos que contenham a query
    relevant = [doc for doc in ctx.deps.documents if query.lower() in doc.lower()]
    return relevant if relevant else ["Nenhum documento relevante encontrado."]

# Função principal adaptada para Jupyter
async def executar_rag():
    # Criar as dependências com documentos de exemplo
    dependencies = RetrievalDependencies(
        documents=[
            "PydanticAI: framework voltado à criação de agentes de IA, com foco em validação de dados, respostas estruturadas, execução de ferramentas e injeção de dependências.",
            "LangGraph: framework voltado à orquestração de agentes e fluxos de trabalho, permitindo gerenciar estados, controlar etapas de execução, implementar decisões condicionais e ciclos.",
            "O PydanticAI e o LangGraph são frameworks Python utilizados no desenvolvimento de aplicações e agentes de inteligência artificial baseados em modelos de linguagem (LLMs)."
        ]
    )
    
    # Executar o agente
    result = await rag_agent.run("Explique como o usa O LangGraph, juntamente com o PydanticAI.", deps=dependencies)
    
    print("Exemplo RAG:")
    print(f"Resposta: {result.output.answer}")
    print("\nDocumentos recuperados:")
    for i, doc in enumerate(result.output.retrieved_docs):
        print(f"{i+1}. {doc}")
    
    return result

# Em Jupyter, basta chamar com await - sem asyncio.run()
await executar_rag()

Exemplo RAG:
Resposta: Você pode usar **LangGraph para coordenar o fluxo** e **PydanticAI para implementar os agentes de IA** executados em cada etapa.

As duas bibliotecas têm responsabilidades diferentes:

| Biblioteca | Papel |
|---|---|
| **LangGraph** | Define etapas, transições, ramificações, ciclos e estado compartilhado. |
| **PydanticAI** | Faz chamadas ao modelo, disponibiliza ferramentas e valida saídas estruturadas com Pydantic. |

A integração pode ser feita diretamente: **um nó assíncrono do LangGraph chama `await agente.run(...)` e devolve uma atualização do estado**. Não é necessário transformar o agente em um agente do LangChain.

## Exemplo: planejar e depois redigir uma resposta

Instale as dependências:

```bash
pip install langgraph pydantic-ai
```

Configure a variável de ambiente `OPENAI_API_KEY` para o modelo usado neste exemplo.

```python
import asyncio
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from pydantic import BaseMo

AgentRunResult(output=RAGResult(answer='Você pode usar **LangGraph para coordenar o fluxo** e **PydanticAI para implementar os agentes de IA** executados em cada etapa.\n\nAs duas bibliotecas têm responsabilidades diferentes:\n\n| Biblioteca | Papel |\n|---|---|\n| **LangGraph** | Define etapas, transições, ramificações, ciclos e estado compartilhado. |\n| **PydanticAI** | Faz chamadas ao modelo, disponibiliza ferramentas e valida saídas estruturadas com Pydantic. |\n\nA integração pode ser feita diretamente: **um nó assíncrono do LangGraph chama `await agente.run(...)` e devolve uma atualização do estado**. Não é necessário transformar o agente em um agente do LangChain.\n\n## Exemplo: planejar e depois redigir uma resposta\n\nInstale as dependências:\n\n```bash\npip install langgraph pydantic-ai\n```\n\nConfigure a variável de ambiente `OPENAI_API_KEY` para o modelo usado neste exemplo.\n\n```python\nimport asyncio\nfrom typing import TypedDict\n\nfrom langgraph.graph import StateGra